In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import abc, gc, os, random, re, sys, time, traceback
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import stable_baselines3 as sb
from stable_baselines3.common.env_util import make_vec_env
from typing import SupportsFloat, Any
import gymnasium as gym
from gymnasium.core import ActType, ObsType

import faiss
import sklearn
import tqdm
import chime

import torch
from torch.nn import functional as F
from torch.utils import data
from torch import optim

In [18]:
from data_loaders import KuaiRecData, commit_dataset, CommitedDataset, JaggedCommittedDataset, commit_dataset_jagged
from dl_trainer import Trainer, sequence_collate, create_sequence_collator
from rs_algorithm.models import AdaptiveRepresentation
from rs_algorithm.models import TwoTowerRecommender, ThreeTowerRecommenderV2, ConcatInteractionHistoryRepresentationWithUserEmbedInitialization
from rs_algorithm.models.user_models.user_02_sim import DynamicPreferenceUserModel, norm_dot_product_based_relavance_calculation, SimUserModel
from rs_algorithm.system2.environments.env_05_almost_everything import SSREnvironment


In [7]:
data_model = KuaiRecData(root='../data/KuaiRec2.0', initial_specimen='smaller')

In [8]:
d_embed = 128
n_users, d_user_feats = data_model.ratings['user_id'].nunique(), data_model.user_features.shape[1]
n_items, d_item_feats = data_model.ratings['item_id'].nunique(), data_model.item_features.shape[1]

uid2idx = {v: i for i, v in enumerate(np.unique(data_model.ratings['user_id']))}
user_model = AdaptiveRepresentation(n_users, d_user_feats, d_hidden=512, d_embed=d_embed, id2idx=uid2idx)
iid2idx = {v: i for i, v in enumerate(np.unique(data_model.ratings['item_id']))}
item_model = AdaptiveRepresentation(n_items, d_item_feats, d_hidden=512, d_embed=d_embed, id2idx=iid2idx)
dot_embed_model = TwoTowerRecommender(user_model, item_model)

Trainer(dot_embed_model, None).load_model('model_id_trained_embedder_v3_128.pt')
dot_embed_model

TwoTowerRecommender(
  (user_model): AdaptiveRepresentation(
    (embedder): Sequential(
      (0): Embedding(1411, 128)
      (1): Flatten(start_dim=1, end_dim=-1)
    )
    (adaptor): Sequential(
      (0): Linear(in_features=1582, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=512, bias=True)
      (3): ReLU()
      (4): Linear(in_features=512, out_features=512, bias=True)
      (5): ReLU()
      (6): Dropout(p=0.2, inplace=False)
      (7): Linear(in_features=512, out_features=128, bias=True)
    )
    (alpha): Sequential(
      (0): Linear(in_features=1582, out_features=1, bias=True)
      (1): Sigmoid()
    )
    (bias): Embedding(1411, 1)
  )
  (item_model): AdaptiveRepresentation(
    (embedder): Sequential(
      (0): Embedding(3327, 128)
      (1): Flatten(start_dim=1, end_dim=-1)
    )
    (adaptor): Sequential(
      (0): Linear(in_features=429, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, 

In [9]:
for mod in (user_model, item_model):
    for par in mod.parameters():
        par.requires_grad = False

history_model = ConcatInteractionHistoryRepresentationWithUserEmbedInitialization(d_embed, 512)
s1recommender = ThreeTowerRecommenderV2(d_embed, user_model, item_model, history_model)

Trainer(s1recommender, None).load_model('model_recommender_v6.pt')
s1recommender

ThreeTowerRecommenderV2(
  (user_model): AdaptiveRepresentation(
    (embedder): Sequential(
      (0): Embedding(1411, 128)
      (1): Flatten(start_dim=1, end_dim=-1)
    )
    (adaptor): Sequential(
      (0): Linear(in_features=1582, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=512, bias=True)
      (3): ReLU()
      (4): Linear(in_features=512, out_features=512, bias=True)
      (5): ReLU()
      (6): Dropout(p=0.2, inplace=False)
      (7): Linear(in_features=512, out_features=128, bias=True)
    )
    (alpha): Sequential(
      (0): Linear(in_features=1582, out_features=1, bias=True)
      (1): Sigmoid()
    )
    (bias): Embedding(1411, 1)
  )
  (item_model): AdaptiveRepresentation(
    (embedder): Sequential(
      (0): Embedding(3327, 128)
      (1): Flatten(start_dim=1, end_dim=-1)
    )
    (adaptor): Sequential(
      (0): Linear(in_features=429, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=5

In [21]:
index = faiss.IndexFlatIP(d_embed)
idx2iid = {}
with torch.no_grad():
    for idx, iid in enumerate(data_model.ratings['item_id'].unique()):
        item_model.eval()
        iem, *_ = item_model(
            torch.tensor([iid], dtype=torch.int64),
            torch.tensor(data_model.get_item_features(iid)[None, :], dtype=torch.float32))
        index.add(iem.numpy())
        idx2iid[idx] = iid


def get_relevant_items(vector, n=7):
    indices = index.search(vector, n)[1]
    return np.vectorize(idx2iid.get)(indices.flatten())


get_relevant_items(np.random.rand(1, d_embed))

array([6787,  600, 6222,  250, 4416, 8524, 8416])

In [11]:
def get_semi_relevant_items(vector, n=3):
    indices = index.search(vector, 1500 + n)[1]
    return np.vectorize(idx2iid.get)(indices[:, 1500:].flatten())


get_semi_relevant_items(np.random.rand(1, d_embed))

array([5513, 2339, 8513])

In [12]:
def random_perpendicular(v):
    shape = v.shape
    v = v.flatten()
    v = np.asarray(v)
    rand = np.random.randn(*v.shape)  # random vector
    proj = np.dot(rand, v) / np.dot(v, v) * v  # projection of rand onto v
    perp = rand - proj  # subtract projection → perpendicular component
    return perp.reshape(shape)


def get_irrelevant_items(vector, n=7):
    vector = random_perpendicular(vector)
    return get_relevant_items(vector, n)


get_irrelevant_items(np.random.rand(1, d_embed))

array([ 600, 6222, 2183, 9900, 3098, 4148, 4666])

In [13]:
def get_random_items(n=2):
    indices = np.random.randint(0, n_items, size=n)
    return np.vectorize(idx2iid.get)(indices)


get_random_items()

array([4106, 3952])

In [14]:
_e = np.random.rand(1, d_embed)
_r = get_relevant_items(_e, 1)
_s = get_semi_relevant_items(_e, 1)
_i = get_irrelevant_items(_e, 1)
_x = get_random_items(1)

with torch.no_grad():
    for _f in (_r, _s, _i, _x):
        iem, *_ = item_model(
            torch.tensor(_f, dtype=torch.int64),
            torch.tensor(data_model.get_item_features(_f), dtype=torch.float32))
        print((F.normalize(torch.tensor(_e), dim=1) * F.normalize(iem, dim=1)).sum(dim=1))

tensor([0.1227], dtype=torch.float64)
tensor([-0.0443], dtype=torch.float64)
tensor([0.0373], dtype=torch.float64)
tensor([-0.0289], dtype=torch.float64)


torch.Size([1411, 128])
Mean 25.018114 Std 463.4131


In [29]:
def create_env():
    return SSREnvironment(
        data_model,
        d_embed,
        user_model,
        item_model,
        s1recommender,
        index,
        idx2iid,
    )

In [30]:
env = make_vec_env(create_env, n_envs=8)
model = sb.DQN("MlpPolicy", env, verbose=1, learning_rate=1e-3, buffer_size=10000, learning_starts=100, batch_size=32)
model

Using cuda device


In [31]:
model.learn(total_timesteps=3000)

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 27.8     |
|    ep_rew_mean      | 0.302    |
|    exploration_rate | 0.189    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 101      |
|    time_elapsed     | 2        |
|    total_timesteps  | 256      |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 57.1     |
|    n_updates        | 4        |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 31.8     |
|    ep_rew_mean      | 1.61     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 8        |
|    fps              | 100      |
|    time_elapsed     | 3        |
|    total_timesteps  | 304      |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 42.8     |
|    n_updates      

In [32]:
model.save("temp/checkpoints/dqn_system_2")

In [25]:
test_env = create_env()
obs, _ = test_env.reset()

In [26]:
for _ in range(20):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = test_env.step(action)
    print(reward)
    if done or truncated: break

0.9945
0.9970000000000001
0.9935
-0.004
-0.005500000000000001
0.992
-0.010500000000000002
0.987
-0.0155
-0.018
-0.0205
-0.023
0.9745
0.972
-0.030500000000000006
-0.03300000000000001
-0.03550000000000001
-0.03800000000000001
-0.040500000000000015
0.957
